## Practical 8: Artificial Neural Networks (ANN)

The objectives of this practical are:

- To understand the basics of ANN
- To learn how to build fully connected multi-layer netural networks
- To learn the workflow of working with an ANN model
- To get hands-on expereince in using the relevant libraries for ANN, i.e. tensorflow and keras
- To run the notebook on both jupyterhub and Google Colab

You can run this notebook on Google Colab:

https://colab.research.google.com/

It's quite intuitive to use Google Colab, which you can explore easily yourself. You can compare the amount of time used for training and evaluating a DNN on both jupyterhub and Google Colab. You can choose a different hardware accelerator, such as GPU and TPU, on Google Colab.

In this practical, we use libraries such as tensorflow and keras both by Google Research, in additon to a number of libraries that we have already used.

## Tutorial 1. Build an ANN for the iris dataset

In [ ]:
from tensorflow import keras
from keras import layers

In [ ]:
from numpy.random import seed
seed(0) 
from tensorflow import random 
random.set_seed(0) # sets the global random seed

We use the iris dataset provided by sklearn.

In [ ]:
from sklearn.datasets import load_iris

In [ ]:
iris = load_iris()
x = iris['data']
y = iris['target']

print('example x data point:', x[0])
print('examle y data point:', y[0])

import numpy as np
print('num classes:', len(np.unique(y)))
num_classes = len(np.unique(y))
num_features = x.shape[1]
print(x.shape)

### Plot the data points in the dataset

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(x[:50,0], x[:50,1], c='r', label='class 1')
plt.scatter(x[50:100,0], x[50:100,1], c='g', label='class 2')
plt.scatter(x[100:150,0], x[100:150,1], c='b', label='class 3')
plt.legend()

As we can see there are three classes in the dataset. We have used the first two features only in the above plot.

### Converting categorical values into numerical values

The to_categorial function from the utils class in the keras module turns a set of classification labels into a classification output vector which is required in the output layer of an ANN.

In this case the $y$ column has three class lables, represented as 0, 1 and 2, which is converted into three columns, each representing one class label. For example, a row in the converted columns might be [0 0 1] representing class label 2.

This is part of the ML workflow, in which we turns each of the labels into a vector with one element only corresponding to the label being 1 and all the other elements corresponding the other labels being 0s. Refer to the lecture slides and tutorials on data preprocessing about how the conversion can be done, i.e. one-hot encoding.

In [ ]:
print('example y class label before conversion:', y[120]) # show the value of y in row 120 before the conversion
y = keras.utils.to_categorical(y) # the conversion takes place
print('example y class label after conversion:', y[120])

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.3)

### Scale the data

This is also part of the ML workflow, in which the data is scaled.

The train and test datasets need to be scaled separately.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

print('example x values before scaling:', x_train[100]) 
x_train = scaler.fit_transform(x_train)
print('example x values after scaling:', x_train[100])

print('example x values before scaling:', x_test[10]) 
x_test = scaler.fit_transform(x_test)
print('example x values after scaling:', x_test[10]) 

### Build an ANN with sequential dense (i.e. fully connected) layers 

This is an example of how an ANN is built. More layers can be added and more nodes in each layer can be added too.

In this example, we have built an ANN consisting of an input layer, two hidden layers and an output layer. The first hidden layer consists of 12 nodes with all features as input and uses the ReLU activation function. The second hidden layer consists of 8 nodes and 12 inputs from the first hidden layer (the same as the number of nodes in the first hidden layer so there is no need to specify it) and uses the ReLU activation function too. The output layer consists of the same number of nodes as the number of classes (i.e. in this case 3) and uses the softmax activation function.

Note that the parameters we have chosen when compiling the model, i.e. the loss function, the optimizer and the performance metrics. Refer to the lecture slides for further information.

We use the history object generated by the model to track how the model performs through the training. The history object is useful for tracking the performance over time.

In [ ]:
from keras.models import Sequential
from keras.layers import Dense

model = Sequential()
model.add(Dense(12, input_dim = num_features, activation='relu'))
model.add(Dense(8, activation='relu'))
model.add(Dense(num_classes, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer = 'sgd', metrics = ['accuracy']) # specify the loss fuction and metrics
history = model.fit(x_train, y_train, epochs=500) # fit the model to the training dataset with epochs = 500


### Performance Evaluation

In [ ]:
loss, accuracy = model.evaluate(x_test, y_test)
print('loss:', loss, 'accuracy:', accuracy)


In [ ]:
plt.plot(history.history['loss'])
plt.xlabel('epoch')
plt.ylabel('loss')


As we can see the loss of the model is gradually reduced as the number of epochs increases. Each epoch involves one round of feedforward and back propagation so the weights are gradually tuned.

In [ ]:
plt.plot(history.history['accuracy'])
plt.xlabel('epoch')
plt.ylabel('accuracy')


As we can see the accuracy of the model is gradually incrased as the number of epochs increases.

## Tutorial 2. Build an ANN for recognising a digit 

This tutorial is an adapted version of the notebook in Exploring MNIST and Deep Neural Networks by Tyler Elliot Bettilyon.

https://medium.com/tebs-lab/how-to-classify-mnist-digits-with-different-neural-network-architectures-39c75a0f03e3

We use the MNIST dataset which is a classic problem for deep neural networks. The ANN takes an input image (28x28 pixels) of a handwritten digit (0–9) and classifies the image as a digit.

### Prepare the MNIST dataset for ANN

Import the MNIST dataset provided by Keras, and flatten the 28x28 pixel image into a vector of length 784. In addition, convert each class label (0-9) into a one-hot encoded vector.

In [ ]:
from tensorflow import keras
from keras.datasets import mnist
from keras.layers import Dense
from keras.models import Sequential
from matplotlib import pyplot as plt
from random import randint

# Preparing the dataset
# Setup train and test splits
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Making a copy of x_train before flattening it for displaying images later
x_train_drawing = x_train

image_size = 784 # 28 x 28
x_train = x_train.reshape(x_train.shape[0], image_size) 
x_test = x_test.reshape(x_test.shape[0], image_size)

# Convert class vectors to binary class matrices
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

### Display an 8x8 table of randomly selected digists

You can run this cell several times to see different sets of randomly selected digit images. Each run shows a table of randomly selected digits.

In [ ]:
for i in range(64):
    ax = plt.subplot(8, 8, i+1)
    ax.axis('off')
    plt.imshow(x_train_drawing[randint(0, x_train.shape[0])], cmap='Greys')

### Build a fully-connected neural netowrk with one hidden layer

In [ ]:
model = Sequential()

# The input layer requires the special input_shape parameter which should match
# the shape of the training data.
model.add(Dense(units=32, activation='sigmoid', input_shape=(image_size,)))
model.add(Dense(units=num_classes, activation='softmax'))
model.summary()

### Question: Can you calculate the numbers of parameters on each layer?

### Train and evaluate the built network

We use the history object generated by the model to track how the model behaves through it's training. The history object is useful for tracking accuracy over time.

In [ ]:
model.compile(optimizer="sgd", loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(x_train, y_train, batch_size=128, epochs=5, verbose=False, validation_split=.1)
loss, accuracy  = model.evaluate(x_test, y_test, verbose=False)

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['training', 'evaluation'], loc='best')
plt.show()

print(f'Test loss: {loss:.3}')
print(f'Test accuracy: {accuracy:.3}')

### Define two helpful functions

Instead of manually building a fully-connected neural network, we can define a function for doing that. 

The create_dense() function takes an array of dense layer sizes and returns a fully connected neural network with the specified layer sizes. For example, create_dense([32, 64, 128]) will return a fully connected neural network with three hidden layers, the first with 32 nodes, the second with 64 nodes, and the third with 128 nodes.

Note that a global variable image_size is used in the create_dense() function to build the input layer which is the same size as the image vector, i.e. 784. 

All the hidden layers use the sigmoid activation function except the output layer, which uses the softmax activation function.

The evaluate() takes the model, the batch size and the number of epochs, prints a summary of the model, trains the model, and then prints the loss and accuracy. This function runs 5 training epochs and uses a fixed batch-size of 128 inputs per batch. It also uses the MNIST dataset imported from Keras.

In [ ]:
def create_dense(layer_sizes):
    model = Sequential()
    model.add(Dense(layer_sizes[0], activation='sigmoid', input_shape=(image_size,)))

    for s in layer_sizes[1:]:
        model.add(Dense(units = s, activation = 'sigmoid'))

    model.add(Dense(units=num_classes, activation='softmax'))

    return model

def evaluate(model, batch_size=128, epochs=5):
    model.summary()
    model.compile(optimizer='sgd', loss='categorical_crossentropy', metrics=['accuracy'])
    history = model.fit(x_train, y_train, batch_size=batch_size, epochs=epochs, validation_split=.1, verbose=False)
    loss, accuracy  = model.evaluate(x_test, y_test, verbose=False)
    
    plt.plot(history.history['accuracy'])
    plt.plot(history.history['val_accuracy'])
    plt.title('model accuracy')
    plt.ylabel('accuracy')
    plt.xlabel('epoch')
    plt.legend(['training', 'evaluation'], loc='best')
    plt.show()

    print()
    print(f'Test loss: {loss:.3}')
    print(f'Test accuracy: {accuracy:.3} \n')


### Compare models with different numbers of hidden layers

All the hidden layers in each model have 32 nodes. The models have 1, 2, 3 and 4 hidden layers respectively. 

In [ ]:
for layers in range(1, 5):
    model = create_dense([32] * layers)
    evaluate(model)

### Question: What are your observations on the performance statistics produced by different models above?

### Compare the lengths of time to train different sizes of deep neural networks

It's obvious that the more hidden layers in a model, the more time it takes to train the model. But consider whether having more layers actually helps produce better performance.

In [ ]:
model = create_dense([32, 32, 32])
evaluate(model, epochs=40)

It's also obvious that the more nodes each hidden layer in a model has, the longer time is neeeded to train it. But again consider whether a model with more nodes actually produces better peerformance.

In [ ]:
for nodes in [32, 64, 128, 256, 512, 1024, 2048]:
    model = create_dense([nodes])
    evaluate(model)


### Question: What are your observations on the statistics about the different models above?

### What about having more layers and more nodes on each layer in a model?

What heppens when we consider these two factors in a model? Let's fix the number of nodes on each hidden layer to 32 first.

In [ ]:
nodes_per_layer = 32
for layers in [1, 2, 3, 4, 5]:
    model = create_dense([nodes_per_layer] * layers)
    evaluate(model, epochs=10*layers)

### Question: What are your observations on the statistics produced by the different models?

Now let's change the number of nodes on each hidden layer to 128.

In [ ]:
nodes_per_layer = 128
for layers in [1, 2, 3, 4, 5]:
    model = create_dense([nodes_per_layer] * layers)
    evaluate(model, epochs=10*layers)

### Question: What are your observations on the statistics produced by the different models?

### What about having smaller batch sizes?

The batch size is often set at something small by default, such as 32 instances, and is not tuned. The batch size of 32 does work well generally. Sometimes models with several hidden layers need to not only train for longer, but also need more corrections per epoch. By reducing the batch size, we can increase the number of overall corrections that a model gets. We can also ensure that it gets more fine grained information by adjusting to errors over smaller batches.

In [ ]:
model = create_dense([32] * 5)
evaluate(model, batch_size=16, epochs=50)

### Question: What are your observations on the statistics produced by the above model?

## Exercise 1. Build an ANN for the diabetes dataset and evaluate the performance of the model